In [ ]:
!pip install nltk spacy scikit-learn sentence-transformers faiss-cpu transformers -q
!python -m spacy download en_core_web_sm -q 

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")


In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

test = "Hello, world! This is a test sentence. Let's see how it works."
test1 = "Hello, world! This is a test sentence. Let's see how it works."
sentence = sent_tokenize(test)
print(sentence)
print("-" * 200)
word = word_tokenize(test)
print(word)

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

text = """Retrieval-Augmented Generation (RAG) combines the power of large language models 
with external knowledge retrieval. It first retrieves relevant documents, then generates 
a response grounded in those documents."""

words = word_tokenize(text)

filtered_words = [w for w in words if w.lower() not in stop_words]
print(filtered_words)

In [ ]:
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ["running", "retrieval", "generating", "documents", "embedded", "queries"]

print(f"{'Word':<15} {'Stemmed':<15} {'Lemmatized':<15}")
print("-" * 45)
for word in test_words:
    stemmed = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word, pos="v")  # pos='v' for verbs
    print(f"{word:<15} {stemmed:<15} {lemmatized:<15}")

print("\ Lemmatization preserves real words; stemming can produce non-words.")

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load a lightweight sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = ["I am hungry", "I need to eat something", "What is the capital of France?"]

embeddings = model.encode(sentences)
print(f"Embedding shape per sentence: {embeddings[0].shape}")

# Cosine Similarity
sim_matrix = cosine_similarity(embeddings)

print("\nCosine Similarity Matrix:")
df_sim = pd.DataFrame(sim_matrix.round(3), index=sentences, columns=sentences)
print(df_sim)

print("\n💡 Sentences 1 & 2 are semantically close → high similarity.")
print("   Sentence 3 is off-topic → lower similarity with 1 & 2.")

In [ ]:
def chunk_text(text, chunk_size=100, overlap=20):
    """
    Split text into overlapping chunks by word count.

    Args:
        text: Input string
        chunk_size: Words per chunk
        overlap: Words shared between adjacent chunks
    Returns:
        List of text chunks
    """
    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks


# Example document
document = """
Retrieval-Augmented Generation (RAG) is an AI framework that enhances large language models 
by connecting them to external knowledge sources. Instead of relying solely on the knowledge 
baked into its parameters during training, a RAG system retrieves relevant documents from a 
knowledge base at inference time and uses those documents to generate more accurate, grounded, 
and up-to-date responses. The retrieval step uses semantic search powered by vector embeddings, 
and the generation step uses a language model conditioned on the retrieved context. 
This makes RAG especially powerful for question answering, summarization, and enterprise 
search applications where factual accuracy and freshness of information are critical.
"""

chunks = chunk_text(document, chunk_size=30, overlap=5)
print(f"📄 Document split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()